# Mission 6 - ps06_solution.ipynb
**The Autonomous Warp State Machine & Deliberation Loop**  
USSE Noosphere | Ship Anomaly 06

Run from the folder that contains the four station folders. Standard library only.

In [1]:
import subprocess, sys
def run_tests(folder):
    r = subprocess.run([sys.executable, "-m", "unittest", "-v"], cwd=folder, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-900:])
def run_script(folder, script):
    r = subprocess.run([sys.executable, script], cwd=folder, capture_output=True, text=True)
    print(r.stdout, r.stderr)

## [1] Systems Architect - AgentStateMachine core
Tests: legal path, illegal events rejected without changing state, terminal states accept only reset, halt/fail from every live state, exhaustive (state, event) check.

In [2]:
run_tests("04_Systems_Architect")

in_edges) ... ok
test_exhaustive_every_state_event_pair (test_state_machine.TestStateMachine.test_exhaustive_every_state_event_pair) ... ok
test_halt_and_fail_from_every_non_terminal_state (test_state_machine.TestStateMachine.test_halt_and_fail_from_every_non_terminal_state) ... ok
test_happy_path_with_one_tool (test_state_machine.TestStateMachine.test_happy_path_with_one_tool) ... ok
test_illegal_event_raises_and_state_is_unchanged (test_state_machine.TestStateMachine.test_illegal_event_raises_and_state_is_unchanged) ... ok
test_terminal_states_only_accept_reset (test_state_machine.TestStateMachine.test_terminal_states_only_accept_reset) ... ok
test_tool_error_returns_to_observation_not_failure (test_state_machine.TestStateMachine.test_tool_error_returns_to_observation_not_failure) ... ok

----------------------------------------------------------------------
Ran 18 tests in 0.003s

OK



## [2] Protocol Specialist - Tool Protocol validator and error-boundary dispatcher
Tests: schema validation (types, ranges, enums, missing/unexpected arguments, bool-is-not-a-number), JSON-RPC error codes, tool exceptions contained, output truncation.

In [3]:
run_tests("02_Protocol_Specialist")

ss) ... ok
test_tool_exception_is_contained (test_tool_protocol.TestDispatcher.test_tool_exception_is_contained) ... ok
test_unknown_tool_code (test_tool_protocol.TestDispatcher.test_unknown_tool_code) ... ok
test_arguments_must_be_object (test_tool_protocol.TestValidator.test_arguments_must_be_object) ... ok
test_bool_is_not_a_number_or_integer (test_tool_protocol.TestValidator.test_bool_is_not_a_number_or_integer) ... ok
test_integer_rejects_float (test_tool_protocol.TestValidator.test_integer_rejects_float) ... ok
test_missing_required_and_unexpected (test_tool_protocol.TestValidator.test_missing_required_and_unexpected) ... ok
test_type_range_enum_length (test_tool_protocol.TestValidator.test_type_range_enum_length) ... ok
test_valid (test_tool_protocol.TestValidator.test_valid) ... ok

----------------------------------------------------------------------
Ran 14 tests in 0.002s

OK



## [3] Data Specialist - MDP trajectory logger and discounted-return evaluator

In [4]:
run_tests("01_Data_Specialist")
run_script("01_Data_Specialist", "evaluation_demo.py")

erAndEval.test_evaluate_statistics) ... ok
test_first_visit_values (test_mdp_logger.TestLoggerAndEval.test_first_visit_values) ... ok
test_gamma_sweep_length (test_mdp_logger.TestLoggerAndEval.test_gamma_sweep_length) ... ok
test_jsonl_round_trip (test_mdp_logger.TestLoggerAndEval.test_jsonl_round_trip) ... ok
test_empty (test_mdp_logger.TestReturns.test_empty) ... ok
test_gamma_zero_is_first_reward_and_gamma_one_is_sum (test_mdp_logger.TestReturns.test_gamma_zero_is_first_reward_and_gamma_one_is_sum) ... ok
test_hand_computed_return (test_mdp_logger.TestReturns.test_hand_computed_return) ... ok
test_matches_explicit_power_sum (test_mdp_logger.TestReturns.test_matches_explicit_power_sum) ... ok
test_returns_to_go_recurrence (test_mdp_logger.TestReturns.test_returns_to_go_recurrence) ... ok

----------------------------------------------------------------------
Ran 11 tests in 0.005s

OK

Rewards: [-1, -1, 10]
  gamma=0.0: G0 = -1.000
  gamma=0.5: G0 = 1.000
  gamma=0.9: G0 = 6.200
  ga

## [4] Security Sentinel - Floyd cycle detector and halting guardrails (Ship Anomaly 06)

In [5]:
run_tests("03_Security_Sentinel")
run_script("03_Security_Sentinel", "anomaly_06_demo.py")

two_identical_steps_are_not_a_loop_by_default (test_cycle_detector.TestGuard.test_two_identical_steps_are_not_a_loop_by_default) ... ok
test_two_repeats_suffice_when_configured_for_deterministic_policies (test_cycle_detector.TestGuard.test_two_repeats_suffice_when_configured_for_deterministic_policies) ... ok
test_deterministic_trace_uses_floyd (test_cycle_detector.TestTrace.test_deterministic_trace_uses_floyd) ... ok
test_nondeterministic_trace_falls_back_to_tail_period (test_cycle_detector.TestTrace.test_nondeterministic_trace_falls_back_to_tail_period) ... ok
test_nondeterministic_without_repetition (test_cycle_detector.TestTrace.test_nondeterministic_without_repetition) ... ok
test_tail_period_needs_full_repeats (test_cycle_detector.TestTrace.test_tail_period_needs_full_repeats) ... ok

----------------------------------------------------------------------
Ran 18 tests in 0.005s

OK

Anomaly 06: AURA repeats run_diagnostic(warp_core) forever without progress.

guard OFF (stops only

## Full squad - end-to-end deliberation loop
Five scenarios (good plan, sloppy plan that recovers, Anomaly 06 loop with and without the guard, false success claim) and a 200-episode random-policy evaluation.

In [6]:
run_script("04_Systems_Architect", "run_mission.py")

scenario                           outcome     halt reason              calls errs  reward  G(0.95)
A good plan                        SUCCESS     None                         4    0    54.0    44.44
B sloppy plan, recovers            SUCCESS     None                         7    3    38.0    22.79
C Ship Anomaly 06 loop (guard on)  HALTED      SHIP_ANOMALY_06_CYCLE        4    0   -22.0   -18.86
D same loop, guard OFF (cap 60)    HALTED      HARD_CAP                    60    0   -78.0   -18.00
E false success claim              FALSE_CLAIM None                         0    0   -10.0   -10.00

State path of scenario C (loop, guard on):
  step 0: f=0.0|diag=000|eng=0 --run_diagnostic({"system": "warp_core"})--> f=0.0|diag=100|eng=0  reward=1.0  
  step 1: f=0.0|diag=100|eng=0 --run_diagnostic({"system": "warp_core"})--> f=0.0|diag=100|eng=0  reward=-1.0  
  step 2: f=0.0|diag=100|eng=0 --run_diagnostic({"system": "warp_core"})--> f=0.0|diag=100|eng=0  reward=-1.0  
  step 3: f=0.0|diag=